# -------------------------------------------------
Predict on new data
-------------------------------------------------

## Imports

In [2]:
import os
import glob
import random
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import time
import math
import requests
from io import BytesIO

# torchvision transforms functional for rotate, adjust_brightness, etc.
from torchvision.transforms import functional as TF  
from torchvision.transforms.functional import InterpolationMode

# torch.nn.functional for interpolate, conv, etc.
import torch.nn.functional as F


## Recreate the model using saved weights

In [3]:
# Device
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

# Model architecture
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# GitHub raw URL
url = "https://github.com/Cebulva/rooftop-solar-analysis-ml-pvlib/blob/main/models/custom_ds_roof_model.pth"

# Download and load weights
response = requests.get(url)
response.raise_for_status()
checkpoint = torch.load(BytesIO(response.content), map_location=device)

model.load_state_dict(checkpoint)
model.eval()



UnpicklingError: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: 

Unsupported operand 10

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

## Prediction on aerial images

In [3]:
# Aerial image download function
# ---------------------------
def get_aerial_image_tensor(lat=53.5625, lon=9.9630, zoom=18, size=400, device=None, show=True, imagenet_norm=False):
    """
    Download a 400x400 aerial image of a residential area and return as a PyTorch tensor.
    """

    if device is None:
        if torch.backends.mps.is_available():
            device = 'mps'
        elif torch.cuda.is_available():
            device = 'cuda'
        else:
            device = 'cpu'

    def latlon_to_tile(lat, lon, zoom):
        n = 2 ** zoom
        xtile = (lon + 180.0) / 360.0 * n
        ytile = (1.0 - math.log(math.tan(math.radians(lat)) +
                                1 / math.cos(math.radians(lat))) / math.pi) / 2.0 * n
        return int(math.floor(xtile)), int(math.floor(ytile))

    x, y = latlon_to_tile(lat, lon, zoom)

    tiles = [(x, y), (x+1, y), (x, y+1), (x+1, y+1)]
    images = []

    for tx, ty in tiles:
        url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{ty}/{tx}"
        success = False
        for attempt in range(3):
            try:
                resp = requests.get(url, timeout=3)
                resp.raise_for_status()
                img_tile = Image.open(BytesIO(resp.content)).convert('RGB')
                images.append(np.array(img_tile))
                success = True
                break
            except requests.RequestException as e:
                print(f"Attempt {attempt+1} failed for tile {tx},{ty}: {e}")
                time.sleep(0.5)
        if not success:
            if zoom > 17:
                return get_aerial_image_tensor(lat, lon, zoom=zoom-1, size=size,
                                               device=device, show=show, imagenet_norm=imagenet_norm)
            else:
                return None

    top = np.concatenate([images[0], images[1]], axis=1)
    bottom = np.concatenate([images[2], images[3]], axis=1)
    full_img = np.concatenate([top, bottom], axis=0)

    start_h = (full_img.shape[0] - size) // 2
    start_w = (full_img.shape[1] - size) // 2
    img_cropped = full_img[start_h:start_h+size, start_w:start_w+size]

    # --- SAME PREPROCESSING AS RoofDataset ---
    img = img_cropped.astype(np.float32) / 255.0
    tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

    if imagenet_norm:
        mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
        tensor = (tensor - mean) / std
    # ----------------------------------------

    return tensor


In [ ]:
# List of coordinates to test
# ---------------------------
coords = [
    (53.63171366, 10.07147466),
    (53.63206781, 10.06995010),
    (53.63485210, 10.09085154),
    (53.63502791, 10.09032893),
]

model.eval()

for i, (lat, lon) in enumerate(coords, start=1):
    print(f"\n=== Point {i}: lat={lat}, lon={lon} ===")

    # Download aerial image (model input, with ImageNet norm)
    img_tensor = get_aerial_image_tensor(
        lat=lat,
        lon=lon,
        zoom=20,
        size=400,
        device=device,
        show=True,
        imagenet_norm=True
    )

    # Download the same image WITHOUT ImageNet normalization (for humans)
    img_tensor_raw = get_aerial_image_tensor(
        lat=lat,
        lon=lon,
        zoom=20,
        size=400,
        device=device,
        show=False,
        imagenet_norm=False
    )

    # Skip if download failed
    if img_tensor is None or img_tensor_raw is None:
        print("Download failed, skipping this point.")
        continue

    # Predict roof mask
    with torch.no_grad():
        logits = model(img_tensor)
        probs  = torch.sigmoid(logits)
        mask   = probs.squeeze().cpu().numpy()

    # Convert tensors for plotting
    input_norm = img_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    input_raw  = img_tensor_raw.squeeze().permute(1, 2, 0).cpu().numpy()

    # Visualization: 4 images in one grid
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(input_norm)
    plt.axis("off")
    plt.title("Model Input (ImageNet-normalized)")

    plt.subplot(1, 4, 2)
    plt.imshow(mask, cmap="gray")
    plt.axis("off")
    plt.title("Predicted Roof Probability")

    plt.subplot(1, 4, 3)
    plt.imshow(input_raw)
    plt.axis("off")
    plt.title("Input Aerial Image (No Normalization)")

    plt.subplot(1, 4, 4)
    plt.imshow(input_raw)
    plt.imshow(mask, cmap="Reds", alpha=0.4)
    plt.axis("off")
    plt.title("Overlay: Prediction on Aerial Image")

    plt.tight_layout()
    plt.show()
